# Chapter 23: Minimum Spanning Trees

The programs from Chapter 23 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## Kruskal's Algorithm

`mst.py`

In [1]:
"""Minimum spanning trees: connect every vertex of a weighted,
undirected graph with the least total edge weight."""

import heapq

from union_find import UnionFind  # a copy of Chapter 19's


def kruskal(n, edges):
    """Vertices 0..n-1; edges as (weight, u, v). Take edges from the
    lightest up, skipping any that would close a loop."""
    uf, tree = UnionFind(n), []
    for weight, u, v in sorted(edges):
        if uf.union(u, v):
            tree.append((weight, u, v))
            if len(tree) == n - 1:
                break
    return tree


def prim(graph, start):
    """Graph: vertex -> {neighbor: weight}. Grow one tree from start,
    always adding the lightest edge that reaches a new vertex."""
    in_tree, tree = {start}, []
    heap = [(w, start, v) for v, w in graph[start].items()]
    heapq.heapify(heap)
    while heap and len(in_tree) < len(graph):
        weight, u, v = heapq.heappop(heap)
        if v in in_tree:
            continue
        in_tree.add(v)
        tree.append((weight, u, v))
        for x, w in graph[v].items():
            if x not in in_tree:
                heapq.heappush(heap, (w, v, x))
    return tree

## The Fiber Network

`network_demo.py`

In [2]:
"""Lay fiber between six (fictional) offices as cheaply as possible.
Costs are in thousands of dollars."""

from mst import kruskal, prim

offices = ["HQ", "Depot", "Lab", "Shop", "Mill", "Port"]
costs = [
    (7, 0, 1),
    (5, 0, 3),
    (8, 1, 2),
    (9, 1, 3),
    (7, 1, 4),
    (5, 2, 4),
    (15, 3, 4),
    (6, 3, 5),
    (8, 4, 5),
    (11, 0, 5),
]
tree = kruskal(len(offices), costs)
for cost, a, b in tree:
    print(f"{offices[a]:>5} -- {offices[b]:<5} {cost}")
print(
    "total:", sum(c for c, _, _ in tree), "of", sum(c for c, _, _ in costs)
)

graph = {i: {} for i in range(len(offices))}
for c, a, b in costs:
    graph[a][b] = graph[b][a] = c
print("Prim's total:", sum(c for c, _, _ in prim(graph, 0)))

   HQ -- Shop  5
  Lab -- Mill  5
 Shop -- Port  6
   HQ -- Depot 7
Depot -- Mill  7
total: 30 of 81
Prim's total: 30


## Timing

`bench_mst.py`

In [3]:
"""Kruskal against Prim on random connected graphs: n vertices, 5n
edges."""

import random

import bench
from mst import kruskal, prim

rng = random.Random(2026)


def make(n):
    edges = [(rng.randint(1, 1000), i, i + 1) for i in range(n - 1)]
    edges += [
        (rng.randint(1, 1000), rng.randrange(n), rng.randrange(n))
        for _ in range(4 * n)
    ]
    graph = {v: {} for v in range(n)}
    for w, u, v in edges:
        if u != v:
            graph[u][v] = graph[v][u] = w
    return edges, graph


sizes = [2_000, 4_000, 8_000, 16_000]
built = {n: make(n) for n in sizes}
results = {
    "kruskal": bench.doubling(kruskal, lambda n: (n, built[n][0]), sizes),
    "prim": bench.doubling(prim, lambda n: (built[n][1], 0), sizes),
}
bench.table(results)

         n         kruskal            prim
     2,000         4.59 ms         7.57 ms
     4,000    11.1 ms x2.4    20.8 ms x2.8
     8,000    30.8 ms x2.8    53.9 ms x2.6
    16,000    76.2 ms x2.5     159 ms x2.9
